# Importing Model From Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="Sandip10/tf_qf_nnlb_vatex_train_ckpt",
    filename="checkpoint-8000/model.safetensors"
)

print(file_path)

In [ ]:
! pip install av 

! pip install sacrebleu rouge-score pycocoevalcap evaluate

! pip install bert_score

## Import Libraries and Configuration


In [ ]:
import os
import re
import json
import torch
import av
import numpy as np
import math
from typing import List, Dict, Tuple

from tqdm import tqdm
import sacrebleu as scb
import evaluate
from pycocoevalcap.cider.cider import Cider

from transformers import AutoImageProcessor

# For model.safetensors
from safetensors.torch import load_file

from model import *


## Path Configuartion

In [ ]:
DatasetFolder = fr"/kaggle/input/datasets/sandipsanjel/vatex-nepali/VATEX_ne"
TEST_VIDEOS_DIR   = os.path.join(DatasetFolder, "videos_test")
TEST_CAPTIONS_FILE = os.path.join(DatasetFolder, "nepali_captions/vatex_test_ne.json")
CHECKPOINT_PATH   =fr"/root/.cache/huggingface/hub/models--Sandip10--tf_qf_nnlb_vatex_train_ckpt/snapshots/8130ab1e4f34246e115d030de1b7d47a0449f309/checkpoint-8000/model.safetensors"
OUTPUT_DIR        = fr"/kaggle/working/evaluation_results"

NUM_FRAMES  = 8
MAX_LENGTH  = 50          
NUM_BEAMS   = 4
BATCH_SIZE  = 2

encoder_name    = "facebook/timesformer-base-finetuned-k600"      
decoder_name    = "facebook/mbart-large-50"                       

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Processing 


In [ ]:
def read_video_pyav(video_path: str, num_frames: int = NUM_FRAMES):
    container = av.open(video_path)
    frames = []
    for frame in container.decode(video=0):
        frames.append(frame.to_rgb().to_ndarray())
    container.close()

    idx    = np.linspace(0, max(len(frames) - 1, 0), num_frames).astype(np.int64)
    frames = [frames[i] for i in idx]
    return frames  # list of (H, W, C) numpy arrays


def process_video(video_path: str, image_processor) -> torch.Tensor:
    frames = read_video_pyav(video_path)
    inputs = image_processor(list(frames), return_tensors="pt")
    return inputs["pixel_values"]   # (1, T, C, H, W)

In [ ]:
def load_test_data(captions_file: str, videos_dir: str):
    
    with open(captions_file, "r", encoding="utf-8") as f:
        data = json.load(f)

   
    if isinstance(data, dict) and "root" in data:
        data = data["root"]

    samples, missing = [], []
    for item in data:
        video_id   = item["video_id"]
        references = item["caption"]
        video_path = os.path.join(videos_dir, f"{video_id}.mp4")
        if not os.path.exists(video_path):
            missing.append(video_id)
            continue
        samples.append({"video_id": video_id, "video_path": video_path, "references": references})

    print(f"Loaded {len(samples)} test videos")
    if missing:
        print(f"  Missing {len(missing)} videos: {missing[:5]}{'...' if len(missing)>5 else ''}")
    return samples

## Inferencing

In [ ]:
@torch.no_grad()
def generate_predictions(model, tokenizer, image_processor, test_samples):
    model.eval()
    predictions, references, video_ids = [], [], []
    failed = []

    ne_lang_id = tokenizer.convert_tokens_to_ids(["npi_Deva"])

    for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Inference"):
        batch = test_samples[i : i + BATCH_SIZE]
        batch_tensors, batch_refs, batch_ids = [], [], []

        for sample in batch:
            try:
                # process_video returns (1, T, C, H, W)
                tensor = process_video(sample["video_path"], image_processor)
                tensor = tensor.squeeze(0)          # (T, C, H, W)
                batch_tensors.append(tensor)
                batch_refs.append(sample["references"])
                batch_ids.append(sample["video_id"])
            except Exception as e:
                print(f"\n  Error on {sample['video_id']}: {e}")
                failed.append(sample["video_id"])

        if not batch_tensors:
            continue

        pixel_values = torch.stack(batch_tensors).to(device)  # (B, T, C, H, W)

        generated_ids = model.generate(
            pixel_values=pixel_values,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            num_return_sequences=NUM_BEAMS,   
            forced_bos_token_id=ne_lang_id,
            no_repeat_ngram_size=3,
            early_stopping=True,
            repetition_penalty=1.3
        )

        # Decode all beams
        all_captions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        #   Debug: print all beams for first batch only
        if i == 0:
            print("\n" + "=" * 60)
            print("  ALL BEAMS — first video in first batch")
            print("=" * 60)
            for j in range(NUM_BEAMS):
                print(f"  beam {j} : {all_captions[j]}")
            print(f"\n  reference 1 : {batch_refs[0][0]}")
            print(f"  reference 2 : {batch_refs[0][1] if len(batch_refs[0]) > 1 else 'N/A'}")
            print("=" * 60 + "\n")

        # keep only beam 0 (best beam) per video 
        # beam 0 for video j is at index j * NUM_BEAMS
        best_captions = [all_captions[j * NUM_BEAMS] for j in range(len(batch_ids))]

        predictions.extend(best_captions)
        references.extend(batch_refs)
        video_ids.extend(batch_ids)

    print(f"\nGenerated {len(predictions)} predictions")
    if failed:
        print(f"  Failed videos : {len(failed)}")
        for vid in failed:
            print(f"    - {vid}")

    return predictions, references, video_ids

## Metrics

In [ ]:
_DEVANAGARI_PUNCT = re.compile(r"([।॥,.!?;:\"'()\[\]{}])")


def tokenize_nepali(text: str) -> List[str]:
    text = _DEVANAGARI_PUNCT.sub(r" \1 ", text)     # separate punctuation
    text = re.sub(r"\s+", " ", text).strip()        # collapse spaces
    return [t for t in text.split() if t]


_ROUGE_BETA = 1.2


#Compute Precision and Recall for ROUGH-L
def _rouge_l_pr(pred: str, ref: str):
    pt = tokenize_nepali(pred)
    rt = tokenize_nepali(ref)
    if not pt or not rt:
        return 0.0, 0.0

    m, n = len(pt), len(rt)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pt[i - 1] == rt[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs       = dp[m][n]
    precision = lcs / m
    recall    = lcs / n
    return precision, recall



# Compute ROUGE-L
def compute_rouge_l(predictions: List[str], references: List[List[str]]) -> float:
    b2 = _ROUGE_BETA ** 2
    scores = []
    for pred, refs in zip(predictions, references):
        pr_pairs  = [_rouge_l_pr(pred, ref) for ref in refs]
        max_p     = max(p for p, r in pr_pairs)
        max_r     = max(r for p, r in pr_pairs)
        if max_p + max_r == 0:
            scores.append(0.0)
        else:
            # Fβ = (1 + β²) * P * R / (R + β² * P)
            f = ((1 + b2) * max_p * max_r) / (max_r + b2 * max_p)
            scores.append(f)
    return (sum(scores) / len(scores)) * 100



# Compute BLEU
def compute_bleu(
    predictions: List[str],
    references: List[List[str]]
) -> Dict[str, float]:
    max_refs        = max(len(rs) for rs in references)
    padded_r        = [rs + [""] * (max_refs - len(rs)) for rs in references]
    refs_transposed = list(zip(*padded_r))

    bleu4_res = scb.BLEU(max_ngram_order=4).corpus_score(predictions, refs_transposed)
    bp        = bleu4_res.bp
    prec      = bleu4_res.precisions

    def _cumulative_bleu(precisions: List[float], n: int, bp: float) -> float:
        if any(p == 0 for p in precisions[:n]):
            return 0.0
        log_avg = sum(math.log(p / 100) for p in precisions[:n]) / n
        return bp * math.exp(log_avg) * 100

    return {
        "BLEU-1": _cumulative_bleu(prec, 1, bp),
        "BLEU-2": _cumulative_bleu(prec, 2, bp),
        "BLEU-3": _cumulative_bleu(prec, 3, bp),
        "BLEU-4": bleu4_res.score,
        "_refs_transposed": refs_transposed,   
    }


# Compute METEOR
def compute_meteor(
    predictions: List[str],
    references: List[List[str]]
) -> float:
    meteor = evaluate.load("meteor")

    scores = []
    for pred, refs in tqdm(
        zip(predictions, references),
        total=len(predictions),
        desc="  METEOR"
    ):
        # references must be a list-of-lists: [[ref1, ref2, ...]]
        result = meteor.compute(predictions=[pred], references=[refs])
        scores.append(result["meteor"])

    return (sum(scores) / len(scores)) * 100


# Compute CHRF++
def compute_chrf(
    predictions: List[str],
    refs_transposed: List[Tuple[str, ...]]
) -> float:
    return scb.CHRF(word_order=2).corpus_score(predictions, refs_transposed).score


# Compute BERT Score
def compute_bertscore(
    predictions: List[str],
    references: List[List[str]]
) -> float:

    bert = evaluate.load("bertscore")
    f1_per_video = []

    for pred, refs in tqdm(
        zip(predictions, references),
        total=len(predictions),
        desc="  BERTScore"
    ):
        res = bert.compute(
            predictions=[pred] * len(refs),
            references=refs,
            model_type="xlm-roberta-large",
            rescale_with_baseline=True,   # makes scores human-interpretable
            lang="ne",                    # Nepali baseline statistics
        )

        f1_per_video.append(max(res["f1"]))

    return (sum(f1_per_video) / len(f1_per_video)) * 100



# Compute CIDEr
def compute_cider(
    predictions: List[str],
    references: List[List[str]]
) -> float:
    gts = {i: rs             for i, rs in enumerate(references)}
    res = {i: [predictions[i]] for i in range(len(predictions))}
    cider_score, _ = Cider().compute_score(gts, res)
    return cider_score * 100

In [ ]:

def compute_all_metrics(
    predictions: List[str],
    refs_nested: List[List[str]]
) -> Dict[str, float]:
    def _normalise(text: str) -> str:
        return " ".join(tokenize_nepali(text))

    preds_norm = [_normalise(p) for p in predictions]
    refs_norm  = [
        [_normalise(r) for r in rs if r.strip()]
        for rs in refs_nested
    ]

    # drop empty pairs
    clean_p, clean_r = [], []
    for p, rs in zip(preds_norm, refs_norm):
        if p and rs:
            clean_p.append(p)
            clean_r.append(rs)

    if not clean_p:
        print(" No valid prediction-reference pairs found.")
        return {}

    n_videos = len(clean_p)
    n_refs   = sum(len(rs) for rs in clean_r)
    print(f"\nEvaluating {n_videos} videos | {n_refs} total references "
          f"| avg {n_refs / n_videos:.1f} refs/video\n")

    results: Dict[str, float] = {}

    # BLEU
    print("Computing BLEU ...")
    bleu_results = compute_bleu(clean_p, clean_r)
    refs_transposed = bleu_results.pop("_refs_transposed")   # reuse for ChrF++
    results.update(bleu_results)
    print(f"  BLEU-1: {results['BLEU-1']:.2f} | BLEU-4: {results['BLEU-4']:.2f}")

    # ROUGE-L 
    print("Computing ROUGE-L (β=1.2, max-P + max-R, pycocoevalcap standard) ...")
    results["ROUGE-L"] = compute_rouge_l(clean_p, clean_r)
    print(f"  ROUGE-L: {results['ROUGE-L']:.2f}")

    # METEOR 
    print("Computing METEOR (all refs passed together) ...")
    results["METEOR"] = compute_meteor(clean_p, clean_r)
    print(f"  METEOR: {results['METEOR']:.2f}")

    # ChrF++
    print("Computing ChrF++ ...")
    results["ChrF++"] = compute_chrf(clean_p, refs_transposed)
    print(f"  ChrF++: {results['ChrF++']:.2f}")

    # BERTScore
    print("Computing BERTScore (max F1, rescaled, lang=ne) ...")
    try:
        results["BERTScore-F1"] = compute_bertscore(clean_p, clean_r)
    except Exception as e:
        print(f"BERTScore failed: {e}")
        results["BERTScore-F1"] = 0.0
    print(f"  BERTScore-F1: {results['BERTScore-F1']:.2f}")

    # CIDEr
    print("Computing CIDEr ...")
    try:
        results["CIDEr"] = compute_cider(clean_p, clean_r)
    except Exception as e:
        print(f"CIDEr failed: {e}")
        results["CIDEr"] = 0.0
    print(f"  CIDEr: {results['CIDEr']:.2f}")

    # Summary
    results["num_videos"] = n_videos
    results["num_refs"]   = n_refs

    _print_results(results)
    return results


def _print_results(results: Dict[str, float]) -> None:
    """Pretty-print the evaluation results table."""
    n_videos = results["num_videos"]
    n_refs   = results["num_refs"]

    print("\n" + "=" * 60)
    print("  NEPALI VIDEO CAPTIONING — EVALUATION RESULTS")
    print("=" * 60)
    print(f"  Videos evaluated : {n_videos}")
    print(f"  Total references : {n_refs}")
    print(f"  Avg refs / video : {n_refs / n_videos:.1f}")
    print("-" * 60)
    print(f"  BLEU-1           : {results['BLEU-1']:>8.2f}")
    print(f"  BLEU-2           : {results['BLEU-2']:>8.2f}")
    print(f"  BLEU-3           : {results['BLEU-3']:>8.2f}")
    print(f"  BLEU-4           : {results['BLEU-4']:>8.2f}")
    print(f"  ROUGE-L          : {results['ROUGE-L']:>8.2f}  (β=1.2, max-P + max-R, pycocoevalcap)")
    print(f"  METEOR           : {results['METEOR']:>8.2f}  (exact-match only - no Nepali stemmer)")
    print(f"  ChrF++           : {results['ChrF++']:>8.2f}")
    print(f"  BERTScore-F1     : {results['BERTScore-F1']:>8.2f}  (rescaled, max F1, lang=ne)")
    print(f"  CIDEr            : {results['CIDEr']:>8.2f}  (plain CIDEr — CIDEr-D unavailable in pip)")
    print("=" * 60)

In [ ]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Tokenizer
    print("Loading tokenizer ...")
    # tokenizer =MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50", src_lang = "ne_NP", tgt_lang = "ne_NP")
    tokenizer =NllbTokenizerFast.from_pretrained("facebook/nllb-200-distilled-600M", src_lang= "npi_Deva", tgt_lang="npi_Deva" )

    # Image Processor
    print("Loading image processor ...")
    image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")

    # Loading Model
    print(f"Loading model from {CHECKPOINT_PATH} ...")
    model2.decoder.config.tie_word_embeddings = False   

    # ckpt_file = os.path.join(CHECKPOINT_PATH, "model.safetensors")
    # ckpt_file = os.path.join(CHECKPOINT_PATH, "pytorch_model.bin")
    ckpt_file = CHECKPOINT_PATH
    if os.path.exists(ckpt_file):
        state_dict = load_file(ckpt_file, device=str(device))    
        # state_dict = torch.load(ckpt_file, map_location= device)
        missing, unexpected = model2.load_state_dict(state_dict, strict=False)
        if missing:
            print(f"  Missing keys   : {missing[:5]}")
        if unexpected:
            print(f"  Unexpected keys: {unexpected[:5]}")
        print("Checkpoint loaded.")
    else:
        print(f"No checkpoint found at {ckpt_file}. Using untrained model!")

    model2.to(device).eval()

    # Test Data
    print("Loading test data ...")
    test_samples = load_test_data(TEST_CAPTIONS_FILE, TEST_VIDEOS_DIR)
    if not test_samples:
        print("No test samples found. Exiting.")
        return

    # Inference
    predictions, references, video_ids = generate_predictions(model2, tokenizer, image_processor, test_samples)

    if not predictions:
        print("No predictions generated. Exiting.")
        return

    # Compute Metrics
    results = compute_all_metrics(predictions, references)

    # Save Metrics
    with open(os.path.join(OUTPUT_DIR, "results.txt"), "w", encoding="utf-8") as f:

        f.write("=" * 60 + "\n")
        f.write("  EVALUATION RESULTS\n")
        f.write("=" * 60 + "\n")
        f.write(f"  Videos evaluated : {results['num_videos']}\n")
        f.write(f"  Total references : {results['num_refs']}\n")
        f.write(f"  Avg refs/video   : {results['num_refs']/results['num_videos']:.1f}\n")
        f.write("-" * 60 + "\n")
        f.write(f"  BLEU-1 (w/ BP)   : {results['BLEU-1']:>8.2f}\n")
        f.write(f"  BLEU-2 (w/ BP)   : {results['BLEU-2']:>8.2f}\n")
        f.write(f"  BLEU-3 (w/ BP)   : {results['BLEU-3']:>8.2f}\n")
        f.write(f"  BLEU-4 (w/ BP)   : {results['BLEU-4']:>8.2f}\n")
        f.write(f"  ROUGE-L          : {results['ROUGE-L']:>8.2f}\n")
        f.write(f"  METEOR           : {results['METEOR']:>8.2f}\n")
        f.write(f"  ChrF++           : {results['ChrF++']:>8.2f}\n")
        f.write(f"  BERTScore-F1     : {results['BERTScore-F1']:>8.2f}\n")
        f.write(f"  CIDEr            : {results['CIDEr']:>8.2f}\n")
        f.write("=" * 60 + "\n\n")

        # Predictions & References
        f.write("  PREDICTIONS & REFERENCES\n")
        f.write("=" * 60 + "\n")
        for vid, pred, refs in zip(video_ids, predictions, references):
            f.write(f"video_id   : {vid}\n")
            f.write(f"prediction : {pred}\n")
            for i, ref in enumerate(refs, 1):
                f.write(f"reference {i} : {ref}\n")
            f.write("-" * 60 + "\n")

    print(f"\n Saved to {OUTPUT_DIR}/results.txt")

In [ ]:
main()